In [ ]:
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated, Literal
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
import operator

In [ ]:
load_dotenv()

chat_model = ChatOpenAI(model='gpt-4o-mini')

In [ ]:
# Create structured data
class SentimentSchema(BaseModel):
    sentiment: Annotated[Literal['positive', 'negative'], Field(description="The sentiment of the text")]


In [ ]:
class DiagnosisSchema(BaseModel):
    issue_type: Annotated[Literal['UX', 'Performance', 'Bug','Support', 'other'], Field(description="The type of issue")]
    tone: Annotated[Literal['angry', 'frustrated', 'disappointed', 'calm'], Field(description="The tone of the response")]
    urgency: Annotated[Literal['high', 'medium', 'low'], Field(description="The urgency of the response")]

In [ ]:
structured_model = chat_model.as_structured_output_model(SentimentSchema)
diagnosis_model = chat_model.as_structured_output_model(DiagnosisSchema)

In [ ]:
# Create a schema
class SmartReplyState(TypedDict):
    review: str
    sentiment: Literal['positive', 'negative']
    diagnosis: dict
    response: str

In [ ]:
# create graph
graph = StateGraph(SmartReplyState)

In [ ]:
def find_sentiment(state: SmartReplyState):
    prompt = f"Determine the sentiment of the following review: {state['review']}"
    result = structured_model.invoke(prompt)
    return {"sentiment": result.sentiment}


def check_sentiment(state: SmartReplyState) -> Literal['positive_response', 'run_diagnosis']:
    if state['sentiment'] == 'positive':
        return "positive_response"
    else:
        return "run_diagnosis"


def positive_response(state: SmartReplyState):
    prompt = f"Generate a positive response to the following review: {state['review']}"
    response = chat_model.invoke(prompt)
    return {"response": response}


def run_diagnosis(state: SmartReplyState):
    prompt = f"""Diagnose the following negative review: {state['review']} \n\n 
    Return issue_type, tone, and urgency in a structured format."""
    result = diagnosis_model.invoke(prompt)
    return {"diagnosis": result.model_dump()}

def negative_response(state: SmartReplyState):
    prompt = f"Generate a response to the following negative review: {state['review']} with diagnosis: {state['diagnosis']}"
    response = chat_model.invoke(prompt)
    return {"response": response}

In [ ]:
# Create nodes
graph.add_node("find_sentiment", find_sentiment)
graph.add_node("check_sentiment", check_sentiment)
graph.add_node("positive_response", positive_response)
graph.add_node("run_diagnosis", run_diagnosis)
graph.add_node("negative_response", negative_response)



In [ ]:
# Create edges
graph.add_edge(START, "find_sentiment")
graph.add_conditional_edges('find_sentiment', check_sentiment)
graph.add_edge("find_sentiment", "positive_response")
graph.add_edge("find_sentiment", "run_diagnosis")
graph.add_edge("run_diagnosis", "negative_response")
graph.add_edge("positive_response", END)
graph.add_edge("negative_response", END)

In [ ]:
workflow = graph.compile()

init_input = "I recently purchased your product and it has exceeded my expectations. The quality is top-notch and the customer service was excellent. I will definitely recommend it to my friends and family."
result = workflow.invoke({"review": init_input})

print("Final Result:")
print(result)